In [1]:
import pickle
import plotly.graph_objects as go
import os
import numpy as np
from GridMLM_tokenizers import CSGridMLMTokenizer
import torch

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
with open('data/gjt_train_harmony_ids.pickle', 'rb') as handle:
    gjt_train_harmony_ids = pickle.load(handle)
with open('data/gjt_val_harmony_ids.pickle', 'rb') as handle:
    gjt_val_harmony_ids = pickle.load(handle)

with open('data/nott_train_harmony_ids.pickle', 'rb') as handle:
    nott_train_harmony_ids = pickle.load(handle)
with open('data/nott_val_harmony_ids.pickle', 'rb') as handle:
    nott_val_harmony_ids = pickle.load(handle)

In [4]:
with open('data/gjt_train_latent_steps_dict.pickle', 'rb') as handle:
    gjt_train_latent_steps_dict = pickle.load(handle)
with open('data/gjt_val_latent_steps_dict.pickle', 'rb') as handle:
    gjt_val_latent_steps_dict = pickle.load(handle)

with open('data/nott_train_latent_steps_dict.pickle', 'rb') as handle:
    nott_train_latent_steps_dict = pickle.load(handle)
with open('data/nott_val_latent_steps_dict.pickle', 'rb') as handle:
    nott_val_latent_steps_dict = pickle.load(handle)

In [5]:
with open('data/gjt_train_latent_steps_list.pickle', 'rb') as handle:
    gjt_train_latent_steps_list = pickle.load(handle)
with open('data/gjt_val_latent_steps_list.pickle', 'rb') as handle:
    gjt_val_latent_steps_list = pickle.load(handle)

with open('data/nott_train_latent_steps_list.pickle', 'rb') as handle:
    nott_train_latent_steps_list = pickle.load(handle)
with open('data/nott_val_latent_steps_list.pickle', 'rb') as handle:
    nott_val_latent_steps_list = pickle.load(handle)

In [6]:
with open('data/umap/umap_2d_per_layer.pickle', 'rb') as handle:
    umap_2d_per_layer = pickle.load(handle)

In [7]:
print(umap_2d_per_layer.keys())

dict_keys([0, 1, 2, 3, 4, 5, 6, 7])


In [8]:
def flatten_piece_metadata(latent_steps_list, key):
    return torch.cat([
        item[key].reshape(-1)
        for item in latent_steps_list
    ]).cpu().numpy()


gjt_train_piece_ids = flatten_piece_metadata(gjt_train_latent_steps_list, "piece_ids")
gjt_train_step_indices = flatten_piece_metadata(gjt_train_latent_steps_list, "step_indices")

gjt_val_piece_ids = flatten_piece_metadata(gjt_val_latent_steps_list, "piece_ids")
gjt_val_step_indices = flatten_piece_metadata(gjt_val_latent_steps_list, "step_indices")

nott_train_piece_ids = flatten_piece_metadata(nott_train_latent_steps_list, "piece_ids")
nott_train_step_indices = flatten_piece_metadata(nott_train_latent_steps_list, "step_indices")

nott_val_piece_ids = flatten_piece_metadata(nott_val_latent_steps_list, "piece_ids")
nott_val_step_indices = flatten_piece_metadata(nott_val_latent_steps_list, "step_indices")

In [9]:
def make_interactive_plot_for_layer(k):
    fig = go.Figure()

    groups = [
        (
            "GJT train",
            umap_2d_per_layer[k]["gjt_train_2d"],
            gjt_train_harmony_ids,
            gjt_train_piece_ids,
            gjt_train_step_indices,
            "firebrick",
        ),
        (
            "GJT validation",
            umap_2d_per_layer[k]["gjt_val_2d"],
            gjt_val_harmony_ids,
            gjt_val_piece_ids,
            gjt_val_step_indices,
            "fuchsia",
        ),
        (
            "NoTT train",
            umap_2d_per_layer[k]["nott_train_2d"],
            nott_train_harmony_ids,
            nott_train_piece_ids,
            nott_train_step_indices,
            "blue",
        ),
        (
            "NoTT validation",
            umap_2d_per_layer[k]["nott_val_2d"],
            nott_val_harmony_ids,
            nott_val_piece_ids,
            nott_val_step_indices,
            "cyan",
        ),
    ]

    for name, coords, harmony_ids, piece_ids, step_indices, color in groups:
        harmony_ids = np.asarray(harmony_ids).reshape(-1)
        piece_ids = np.asarray(piece_ids).reshape(-1)
        step_indices = np.asarray(step_indices).reshape(-1)

        harmony_tokens = np.asarray([
            tokenizer.ids_to_tokens[int(token_id)]
            for token_id in harmony_ids
        ])

        assert len(coords) == len(harmony_ids) == len(piece_ids) == len(step_indices), (
            f"{name}: coordinates and point metadata differ in length"
        )

        # Columns are harmony token, piece ID, and position within the piece.
        customdata = np.column_stack((harmony_tokens, piece_ids, step_indices))

        fig.add_trace(go.Scattergl(
            x=coords[:, 0],
            y=coords[:, 1],
            mode="markers",
            name=name,
            customdata=customdata,
            marker=dict(color=color, size=5, opacity=0.35),
            hovertemplate=(
                f"{name}<br>"
                "Piece ID: %{customdata[1]}<br>"
                "Harmony: %{customdata[0]}<br>"
                "Position: %{customdata[2]}<br>"
                "UMAP 1: %{x:.3f}<br>"
                "UMAP 2: %{y:.3f}"
                "<extra></extra>"
            ),
        ))

    # This trace is updated by the hover handler; it does not affect marker colors.
    fig.add_trace(go.Scatter(
        x=[],
        y=[],
        mode="lines",
        line=dict(width=2),
        hoverinfo="skip",
        showlegend=False,
    ))
    overlay_index = len(groups)

    post_script = """
    const plot = document.getElementById('{plot_id}');
    const overlayIndex = __OVERLAY_INDEX__;

    plot.on('plotly_hover', (event) => {
        const point = event.points[0];
        if (point.curveNumber >= overlayIndex) return;

        const trace = plot._fullData[point.curveNumber];
        const pieceId = String(point.customdata[1]);

        const pointsInPiece = trace.customdata
            .map((data, i) => ({
                index: i,
                pieceId: String(data[1]),
                step: Number(data[2])
            }))
            .filter(item => item.pieceId === pieceId)
            .sort((a, b) => a.step - b.step);

        console.log({
            trace: point.curveNumber,
            pieceId,
            matchingPoints: pointsInPiece.length,
            firstSteps: pointsInPiece.slice(0, 5).map(item => item.step)
        });

        const x = pointsInPiece.map(item => trace.x[item.index]);
        const y = pointsInPiece.map(item => trace.y[item.index]);

        Plotly.restyle(plot, {
            x: [x],
            y: [y],
            'line.color': [trace.marker.color],
            'line.width': [2]
        }, [overlayIndex])
    });

    plot.on('plotly_unhover', () => {
        Plotly.restyle(plot, {x: [[]], y: [[]]}, [overlayIndex]);
    });
    """.replace("__OVERLAY_INDEX__", str(overlay_index))

    fig.update_layout(
        title=f"UMAP of latent layer {k}",
        xaxis_title="Component 1",
        yaxis_title="Component 2",
        hovermode="closest",
    )

    os.makedirs("html/umap", exist_ok=True)
    fig.write_html(
        f"html/umap/umap_{k}.html",
        include_plotlyjs="cdn",
        post_script=post_script,
    )

In [11]:
for k in range(8):
    make_interactive_plot_for_layer(k)